In [1]:
import pandas as pd
import numpy as nd
import matplotlib as ply
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
!pip install datasets pandas --break-system-packages


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset

dataset = load_dataset("openfoodfacts/product-database", split="food", streaming=True)

# grab just ONE row to inspect its structure
first_row = next(iter(dataset))
print(first_row.keys())

C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jayak\.cache\huggingface\hub\datasets--openfoodfacts--product-database. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyth

dict_keys(['additives_n', 'additives_tags', 'allergens_tags', 'brands_tags', 'brands', 'categories', 'categories_tags', 'categories_properties', 'checkers_tags', 'ciqual_food_name_tags', 'cities_tags', 'code', 'compared_to_category', 'complete', 'completeness', 'correctors_tags', 'countries_tags', 'created_t', 'creator', 'data_quality_errors_tags', 'data_quality_info_tags', 'data_quality_warnings_tags', 'data_sources_tags', 'environmental_score_data', 'environmental_score_grade', 'environmental_score_score', 'environmental_score_tags', 'editors', 'emb_codes_tags', 'emb_codes', 'entry_dates_tags', 'food_groups_tags', 'generic_name', 'images', 'informers_tags', 'ingredients_analysis_tags', 'ingredients_from_palm_oil_n', 'ingredients_n', 'ingredients_original_tags', 'ingredients_percent_analysis', 'ingredients_tags', 'ingredients_text', 'ingredients_with_specified_percent_n', 'ingredients_with_unspecified_percent_n', 'ingredients_without_ciqual_codes_n', 'ingredients_without_ciqual_codes'

In [4]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("openfoodfacts/product-database", split="food", streaming=True)
sample = dataset.take(20000)

rows = []
for item in sample:
    rows.append({
        "code": item.get("code"),
        "product_name": item.get("product_name"),
        "categories": item.get("categories"),
        "ingredients_text": item.get("ingredients_text"),
    })

df = pd.DataFrame(rows)
df = df.dropna(subset=["product_name", "categories"])
df.to_csv("food_facts_sample.csv", index=False)

print(len(df))
df.head()

17989


,code,product_name,categories,ingredients_text
1,0000105000011,"[{'lang': 'main', 'text': 'Chamomile Herbal Te...",null,"[{'lang': 'main', 'text': 'CHAMOMILE FLOWERS.'..."
2,0000105000042,"[{'lang': 'main', 'text': 'Lagg's, herbal tea,...","Plant-based foods and beverages, Beverages, Ho...","[{'lang': 'main', 'text': 'Peppermint.'}, {'la..."
3,0000105000059,"[{'lang': 'main', 'text': 'Linden Flowers Tea'...","Beverages and beverages preparations, Plant-ba...","[{'lang': 'main', 'text': 'LINDEN FLOWERS.'}, ..."
5,0000105000196,"[{'lang': 'main', 'text': 'Apple & Cinnamon Te...",null,"[{'lang': 'main', 'text': 'TEA, CINNAMON & NAT..."
6,0000105000219,"[{'lang': 'main', 'text': 'Green Tea'}, {'lang...",null,"[{'lang': 'main', 'text': 'GREEN TEA.'}, {'lan..."


In [5]:
import pandas as pd

# Load our cleaned dataset back in from where we saved it last time
df = pd.read_csv("food_facts_sample.csv")
print(len(df))
df.head()

17989


,code,product_name,categories,ingredients_text
0,105000011,"[{'lang': 'main', 'text': 'Chamomile Herbal Te...",NaN,"[{'lang': 'main', 'text': 'CHAMOMILE FLOWERS.'..."
1,105000042,"[{'lang': 'main', 'text': ""Lagg's, herbal tea,...","Plant-based foods and beverages, Beverages, Ho...","[{'lang': 'main', 'text': 'Peppermint.'}, {'la..."
2,105000059,"[{'lang': 'main', 'text': 'Linden Flowers Tea'...","Beverages and beverages preparations, Plant-ba...","[{'lang': 'main', 'text': 'LINDEN FLOWERS.'}, ..."
3,105000196,"[{'lang': 'main', 'text': 'Apple & Cinnamon Te...",NaN,"[{'lang': 'main', 'text': 'TEA, CINNAMON & NAT..."
4,105000219,"[{'lang': 'main', 'text': 'Green Tea'}, {'lang...",NaN,"[{'lang': 'main', 'text': 'GREEN TEA.'}, {'lan..."


In [6]:
def extract_text(field):
    if isinstance(field, list):
        for entry in field:
            if entry.get("lang") == "main":
                return entry.get("text")
        if len(field) > 0:
            return field[0].get("text")
    return None

rows = []
for item in sample:
    rows.append({
        "code": item.get("code"),
        "product_name": extract_text(item.get("product_name")),
        "categories": item.get("categories"),
        "ingredients_text": extract_text(item.get("ingredients_text")),
    })

df = pd.DataFrame(rows)

# treat the literal string "null" as missing too, So first we change null to none for python to understand its a empty value and not string called null

df["categories"] = df["categories"].replace("null", None)

df = df.dropna(subset=["product_name", "categories"])
df.to_csv("food_facts_sample.csv", index=False)

print(len(df))
df.head()

16966


,code,product_name,categories,ingredients_text
2,0000105000042,"Lagg's, herbal tea, peppermint","Plant-based foods and beverages, Beverages, Ho...",Peppermint.
3,0000105000059,Linden Flowers Tea,"Beverages and beverages preparations, Plant-ba...",LINDEN FLOWERS.
8,0000105000356,"Lagg's, herbal tea, chamomile * mint","Plant-based foods and beverages, Beverages, Ho...",Chamomile spearmint.
10,0000105000417,"Lagg's, dieter's herbal tea","Plant-based foods and beverages, Beverages, Ho...","Andropogon citratus, uva ursi, hibiscus flower..."
11,0000105200923,"Lagg's, kidneytea, herbal tea","Plant-based foods and beverages, Beverages, Ho...","Shave grass, corn silk, uva ursi, juliana adst..."


In [7]:
df.iloc[11]["ingredients_text"]

'Fortified wheat flour (with calcium, iron, niacin, thiamin), vegetable oil (palm), sugar, wholemeal wheat flour, dried cream cheese (2.5%) (milk), dried skimmed milk, oatmeal, low fat yogurt powder (milk), dried whey (milk), dextrose, raising agents (sodium bicarbonate, ammonium bicarbonate), salt, wheat starch, natural lemon flavouring, colour (beta carotene).'

In [8]:
# JUST TO SEE THE ENTIRE INGREDIENT TEXT
pd.set_option('display.max_colwidth', None)
df.head()

,code,product_name,categories,ingredients_text
2,0000105000042,"Lagg's, herbal tea, peppermint","Plant-based foods and beverages, Beverages, Hot beverages, Plant-based beverages, Teas, Tea bags",Peppermint.
3,0000105000059,Linden Flowers Tea,"Beverages and beverages preparations, Plant-based foods and beverages, Beverages, Hot beverages, Plant-based beverages, Teas, Null, en:tea-bags",LINDEN FLOWERS.
8,0000105000356,"Lagg's, herbal tea, chamomile * mint","Plant-based foods and beverages, Beverages, Hot beverages, Plant-based beverages, Teas, Tea bags",Chamomile spearmint.
10,0000105000417,"Lagg's, dieter's herbal tea","Plant-based foods and beverages, Beverages, Hot beverages, Plant-based beverages, Teas, Tea bags","Andropogon citratus, uva ursi, hibiscus flowers, cinnamon, equisetum arvense, flourensia cernua."
11,0000105200923,"Lagg's, kidneytea, herbal tea","Plant-based foods and beverages, Beverages, Hot beverages, Plant-based beverages, Teas, Tea bags","Shave grass, corn silk, uva ursi, juliana adstringen, boldo, hibiscus flowers, orange blossom."


In [9]:
df = df.reset_index(drop=True)
df.to_csv("food_facts_sample.csv", index=False)
print(len(df))

16966


In [68]:
import pandas as pd

df = pd.read_csv("food_facts_sample.csv")
print(len(df))
df.head(2)

print(df["categories"].isna().sum())

16966
133


#####Here's something important: categories right now often has multiple categories crammed into one string, like "Plant-based foods and beverages, Beverages, Hot beverages". A model can't cleanly predict a messy multi-category string like that — we need one clean label per product.For a baseline model, the simplest fix is: just take the first category listed as the "main" label.

In [69]:
import pandas as pd

# 1. Load raw data
df = pd.read_csv("food_facts_sample.csv")
print(f"Loaded: {len(df)} rows")

# 2. Extract main category from the categories column
df["main_category"] = df["categories"].apply(
    lambda x: x.split(",")[0].strip() if isinstance(x, str) else None
)

# 3. Remove undefined categories
df = df[~df["main_category"].isin(["undefined", "Undefined"])]
print(f"After removing undefined: {len(df)} rows")

# 4. Translate French category names to English
translation_map = {
    "Aliments et boissons à base de végétaux": "Plant-based foods and beverages",
    "Boissons": "Beverages",
    "Viandes et dérivés": "Meats and their products",
}
df["main_category"] = df["main_category"].replace(translation_map)

# 5. Filter to top 10 categories
top10 = df["main_category"].value_counts().head(10).index
df = df[df["main_category"].isin(top10)]
print(f"After top 10 filter: {len(df)} rows")
print(df["main_category"].value_counts())

# 6. Save clean file
df.to_csv('food_facts_clean.csv', index=False)
print("Saved!")

Loaded: 16966 rows
After removing undefined: 7009 rows
After top 10 filter: 6033 rows
main_category
Plant-based foods and beverages    1793
Snacks                             1732
Beverages                           646
Condiments                          586
Dairies                             466
Desserts                            365
Meats and their products            162
Meals                               151
Frozen foods                         68
Sweeteners                           64
Name: count, dtype: int64
Saved!


In [70]:
from sklearn.model_selection import train_test_split


X = df["product_name"]
y = df["main_category"]

# Split into 80% train, 20% test.
# random_state=42 just makes the split reproducible — same split every time we run it.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(len(X_train), len(X_test))

4826 1207


In [71]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TfidfVectorizer will learn the vocabulary from our training text
# and convert each product name into a row of TF-IDF numbers
vectorizer = TfidfVectorizer()

# fit_transform on TRAINING data: it learns the vocabulary AND converts the text, both at once.
X_train_tfidf = vectorizer.fit_transform(X_train)

# transform (not fit_transform) on TEST data: we reuse the SAME vocabulary learned from
# training — we never let the model "see" test data while learning, otherwise it's cheating.
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(4826, 2969)


###### 4,826 training products, and TF-IDF found 3,011 unique words across all their names. Each product name is now represented as a row of 2969 numbers (mostly zeros, since most words don't appear in most product names — that's normal, it's called a "sparse" representation).

In [72]:
from sklearn.linear_model import LogisticRegression

# Create the model. max_iter is raised from the default (100) because with
# 3,011 features, the model sometimes needs more attempts to fully learn — 
# this just avoids a "didn't converge" warning, not a big deal conceptually.
model = LogisticRegression(max_iter=1000)

# Train the model: show it the TF-IDF numbers (X_train_tfidf) 
# alongside the correct answers (y_train), so it can learn the word→category patterns.
model.fit(X_train_tfidf, y_train)

print("Training done")

Training done


In [73]:
# Use the trained model to predict categories for the test set
y_pred = model.predict(X_test_tfidf)

# Compare predictions to the actual correct categories, and print
# an accuracy score plus a detailed breakdown per category.
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8293289146644574
                                 precision    recall  f1-score   support

                      Beverages       0.97      0.82      0.89       142
                     Condiments       0.88      0.84      0.86       108
                        Dairies       0.84      0.83      0.83        82
                       Desserts       0.97      0.84      0.90        68
                   Frozen foods       1.00      0.14      0.25        21
                          Meals       0.73      0.33      0.46        24
       Meats and their products       0.87      0.36      0.51        36
Plant-based foods and beverages       0.75      0.91      0.83       366
                         Snacks       0.84      0.88      0.86       348
                     Sweeteners       0.83      0.42      0.56        12

                       accuracy                           0.83      1207
                      macro avg       0.87      0.64      0.69      1207
                   w

In [74]:
# Randomly pick 30 products from our cleaned dataset to test OCR on.
# random_state=42 makes this reproducible - same 30 products every time we run it.
sample_df = df.sample(30, random_state=42)
sample_df[["code", "product_name", "main_category"]]

,code,product_name,main_category
9436,14113913256,Pistachios Salt & Pepper,Plant-based foods and beverages
9943,15300200029,Creamy Four Cheese,Meals
6014,11213024622,"Spartan, cannellini beans, white kidney beans",Plant-based foods and beverages
29,433906023,Best Sweet-Potato Cookies,Snacks
13065,19473005962,Ice cream,Desserts
9165,13628608930,Federzoni red wine vinegar,Condiments
3595,11150021166,100% Vegetable Juice,Plant-based foods and beverages
11787,17600043191,Candied yams,Plant-based foods and beverages
695,10300000389,In-Shell Mixed Nuts,Snacks
1001,11110268754,"King Soopers, City Market, Smores Soft Top Cookies",Snacks


In [20]:
def barcode_to_image_url(code):
    # Convert to string and pad with leading zeros until it's 13 digits long
    # (Open Food Facts always expects 13-digit barcodes for building the folder path)
    code = str(code).zfill(13)
    
    # Split the first 9 digits into 3 groups of 3, and keep the remaining 4 digits as the last folder
    # e.g. "0011110018342" -> "001/111/001/8342"
    path = f"{code[0:3]}/{code[3:6]}/{code[6:9]}/{code[9:]}"
    
    return f"https://images.openfoodfacts.org/images/products/{path}/1.jpg"

# Rebuild the image_url column with the corrected function
sample_df["image_url"] = sample_df["code"].apply(barcode_to_image_url)
sample_df[["code", "product_name", "image_url"]]

,code,product_name,image_url
9436,14113913256,Pistachios Salt & Pepper,https://images.openfoodfacts.org/images/products/001/411/391/3256/1.jpg
9943,15300200029,Creamy Four Cheese,https://images.openfoodfacts.org/images/products/001/530/020/0029/1.jpg
6014,11213024622,"Spartan, cannellini beans, white kidney beans",https://images.openfoodfacts.org/images/products/001/121/302/4622/1.jpg
29,433906023,Best Sweet-Potato Cookies,https://images.openfoodfacts.org/images/products/000/043/390/6023/1.jpg
13065,19473005962,Ice cream,https://images.openfoodfacts.org/images/products/001/947/300/5962/1.jpg
9165,13628608930,Federzoni red wine vinegar,https://images.openfoodfacts.org/images/products/001/362/860/8930/1.jpg
3595,11150021166,100% Vegetable Juice,https://images.openfoodfacts.org/images/products/001/115/002/1166/1.jpg
11787,17600043191,Candied yams,https://images.openfoodfacts.org/images/products/001/760/004/3191/1.jpg
695,10300000389,In-Shell Mixed Nuts,https://images.openfoodfacts.org/images/products/001/030/000/0389/1.jpg
1001,11110268754,"King Soopers, City Market, Smores Soft Top Cookies",https://images.openfoodfacts.org/images/products/001/111/026/8754/1.jpg


## THE ABOVE URLS DO NOT WORK

In [21]:
import requests

# Open Food Facts asks that requests include a descriptive User-Agent header,
# otherwise it may block or return an empty response.
headers = {"User-Agent": "AnugrahaProductClassifier/1.0 (test run)"}

response = requests.get(
    "https://world.openfoodfacts.org/api/v2/product/3017624010701.json",
    headers=headers
)

# Print status code and raw text first, so we can see exactly what came back
# before trying to parse it as JSON.
print(response.status_code)
print(response.text[:300])

200
{"code":"3017624010701","product":{"_id":"3017624010701","_keywords":["brotaufstriche","ferrero","fr-pâte","frühstücke","glutenfrei","haselnussaufstriche","haselnusscreme","nougatcreme","nutella","other","schoko","süße","tartiner","und"],"added_countries_tags":[],"additives_n":0,"additives_original_


In [22]:
import requests

headers = {"User-Agent": "AnugrahaProductClassifier/1.0 (test run)"}

response = requests.get(
    "https://world.openfoodfacts.org/api/v2/product/3017624010701.json",
    headers=headers
)

data = response.json()

# Now that we know the response is valid JSON, pull out just the image_url field
print(data["product"].get("image_url"))

https://images.openfoodfacts.org/images/products/301/762/401/0701/front_en.100.400.jpg


In [75]:
import requests
import time

headers = {"User-Agent": "AnugrahaProductClassifier/1.0 (test run)"}

def get_real_image_url(code):
    # Ask Open Food Facts's API directly for this product's data
    url = f"https://world.openfoodfacts.org/api/v2/product/{code}.json"
    response = requests.get(url, headers=headers)
    
    # If the request failed for any reason, just return None instead of crashing
    if response.status_code != 200:
        return None
    
    data = response.json()
    
    # Some barcodes might not exist in their database at all -
    # in that case "product" won't be present, so we check safely
    product = data.get("product", {})
    return product.get("image_url")

# Apply this function to every barcode in our sample,
# with a tiny pause between requests so we don't overwhelm their server
image_urls = []
for code in sample_df["code"]:
    image_urls.append(get_real_image_url(code))
    time.sleep(0.5)  # half-second pause between requests, politeness/rate-limit safety

sample_df["image_url"] = image_urls
sample_df[["code", "product_name", "image_url"]]

,code,product_name,image_url
9436,14113913256,Pistachios Salt & Pepper,https://images.openfoodfacts.org/images/products/001/411/391/3256/front_en.4.400.jpg
9943,15300200029,Creamy Four Cheese,https://images.openfoodfacts.org/images/products/001/530/020/0029/front_en.15.400.jpg
6014,11213024622,"Spartan, cannellini beans, white kidney beans",NaN
29,433906023,Best Sweet-Potato Cookies,NaN
13065,19473005962,Ice cream,NaN
9165,13628608930,Federzoni red wine vinegar,https://images.openfoodfacts.org/images/products/001/362/860/8930/front_en.6.400.jpg
3595,11150021166,100% Vegetable Juice,NaN
11787,17600043191,Candied yams,https://images.openfoodfacts.org/images/products/001/760/004/3191/front_en.5.400.jpg
695,10300000389,In-Shell Mixed Nuts,NaN
1001,11110268754,"King Soopers, City Market, Smores Soft Top Cookies",NaN


In [24]:
import requests
import time

headers = {"User-Agent": "AnugrahaProductClassifier/1.0 (test run)"}

# Recreate the 150-row sample first, since this is what was missing
bigger_sample = df.sample(150, random_state=42)

def get_real_image_url(code):
    url = f"https://world.openfoodfacts.org/api/v2/product/{code}.json"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None
        data = response.json()
        product = data.get("product", {})
        return product.get("image_url")
    except requests.exceptions.RequestException:
        return None

image_urls = []
for code in bigger_sample["code"]:
    image_urls.append(get_real_image_url(code))
    time.sleep(1)

bigger_sample["image_url"] = image_urls
bigger_sample_clean = bigger_sample[bigger_sample["image_url"].notna()]
print(len(bigger_sample_clean))

28


In [25]:
import requests

# Pick the very first product in our clean list to test with
test_url = bigger_sample_clean["image_url"].iloc[0]
test_code = bigger_sample_clean["code"].iloc[0]
test_name = bigger_sample_clean["product_name"].iloc[0]

# Download the actual image bytes from the URL
response = requests.get(test_url, headers=headers)

# Save those bytes to a real file on disk, named after the product's barcode
with open(f"{test_code}.jpg", "wb") as f:
    f.write(response.content)

print("Saved:", f"{test_code}.jpg")
print("This product's real name is:", test_name)

Saved: 14113913256.jpg
This product's real name is: Pistachios Salt & Pepper


In [26]:
import os

# Create a subfolder called "openfoodFACT_images" inside your current working folder
# (which is already MY PYTHONNNNNNNN, since that's where the notebook is running from)
folder_name = "openfoodFACT_images"
os.makedirs(folder_name, exist_ok=True)

print("Folder ready at:", os.path.abspath(folder_name))

Folder ready at: C:\Users\jayak\MY PYTHONNNNNNNN\openfoodFACT_images


In [27]:
import requests

test_url = bigger_sample_clean["image_url"].iloc[0]
test_code = bigger_sample_clean["code"].iloc[0]

response = requests.get(test_url, headers=headers)

# Build the save path: folder_name + filename, combined safely
save_path = os.path.join(folder_name, f"{test_code}.jpg")

with open(save_path, "wb") as f:
    f.write(response.content)

print("Saved to:", save_path)

Saved to: openfoodFACT_images\14113913256.jpg


In [28]:
import requests

# Loop through every row in our clean 30-product table
for index, row in bigger_sample_clean.iterrows():
    url = row["image_url"]
    code = row["code"]
    
    # Download this specific product's image
    response = requests.get(url, headers=headers)
    
    # Save it into our folder, named by its barcode
    save_path = os.path.join(folder_name, f"{code}.jpg")
    with open(save_path, "wb") as f:
        f.write(response.content)

print("Downloaded all images into:", folder_name)

Downloaded all images into: openfoodFACT_images


In [29]:
pip install easyocr --break-system-packages

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [81]:
import easyocr
import os

# Create the OCR reader for English text
# (first run downloads a pretrained model - only happens once, then it's cached)
reader = easyocr.Reader(['en'])

# Point to our test image, now living inside openfoodFACT_images
test_image_path = os.path.join(folder_name, "19458000180.jpg")

# Run OCR - this returns a list of (location, text, confidence) for each detected text region
result = reader.readtext(test_image_path)

# Print just the text and confidence for each detection
for detection in result:
    text = detection[1]
    confidence = detection[2]
    print(f"Text: {text}  |  Confidence: {confidence:.2f}")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Text: TORTILLA  |  Confidence: 0.79
Text: CHIPS  |  Confidence: 0.98
Text: ROUNDS  |  Confidence: 0.98
Text: Prrobebtives  |  Confidence: 0.10
Text: Nate  |  Confidence: 0.16
Text: 02 (397 Qi  |  Confidence: 0.28


In [31]:
# Only keep detected text where confidence is 0.65 or higher
CONFIDENCE_THRESHOLD = 0.65

filtered_text = [detection[1] for detection in result if detection[2] >= CONFIDENCE_THRESHOLD]

# Join the surviving words into one string, same shape as our product_name column
extracted_text = " ".join(filtered_text)
print(extracted_text)

TORTILLA CHIPS ROUNDS


In [82]:
# Convert the OCR text using our ALREADY-TRAINED vectorizer (transform, not fit_transform)
ocr_tfidf = vectorizer.transform([extracted_text])

# Predict the category using our ALREADY-TRAINED model
predicted_category = model.predict(ocr_tfidf)

print("OCR extracted text:", extracted_text)
print("Predicted category:", predicted_category[0])
print("Actual known category: Snacks (Tortilla Chips)")

OCR extracted text: Barbecue BOriginal Sauce
Predicted category: Condiments
Actual known category: Snacks (Tortilla Chips)


In [83]:
import easyocr
import os

CONFIDENCE_THRESHOLD = 0.65

# Lists to collect our results for all 30 products
ocr_texts = []
predicted_categories = []

# Loop through every row in our clean 30-product table
for index, row in bigger_sample_clean.iterrows():
    code = row["code"]
    image_path = os.path.join(folder_name, f"{code}.jpg")
    
    # Run OCR on this product's saved image
    result = reader.readtext(image_path)
    
    # Keep only text detections with confidence >= 0.65, join into one string
    filtered_text = [detection[1] for detection in result if detection[2] >= CONFIDENCE_THRESHOLD]
    extracted_text = " ".join(filtered_text)
    
    # If OCR found NOTHING usable, extracted_text will be an empty string ""
    # We handle that safely rather than letting it break the model
    if extracted_text.strip() == "":
        predicted = "No text detected"
    else:
        ocr_tfidf = vectorizer.transform([extracted_text])
        predicted = model.predict(ocr_tfidf)[0]
    
    ocr_texts.append(extracted_text)
    predicted_categories.append(predicted)

# Attach both new columns to our table
bigger_sample_clean["ocr_text"] = ocr_texts
bigger_sample_clean["predicted_category"] = predicted_categories

bigger_sample_clean[["product_name", "main_category", "ocr_text", "predicted_category"]]

C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  s

,product_name,main_category,ocr_text,predicted_category
9436,Pistachios Salt & Pepper,Plant-based foods and beverages,Salt & Pepper,Condiments
11787,Candied yams,Plant-based foods and beverages,CANDIED YMS CUt SWEET POTATOES SYRUP SIMMERED,Plant-based foods and beverages
11498,Sharp cheddar cheese ball,Dairies,SPREADABLE C 0 'CHEDDAR,Dairies
13454,"Clabber Girl, Corn Starch",Plant-based foods and beverages,CLABBER GIRL CORN STARCH THICKENS,Plant-based foods and beverages
11036,Crunchy Granola Bars Oats 'n Honey,Snacks,CRUNCHY Oats Honey 98,Plant-based foods and beverages
11860,Jamon Serrano,Meats and their products,@npofrio,Plant-based foods and beverages
10967,Gushers Strawberry Splash and Tropical Fruit 6 Count,Snacks,VARIETY PACK,Snacks
281,Hamburger Dill Chips,Plant-based foods and beverages,HAMBURGER CHIPS AWsh SINCE 'DILL Make,Snacks
2915,Coffee Creamer,Plant-based foods and beverages,VANILLA,Snacks
13753,Desiccated Coconut,Plant-based foods and beverages,by Sainsburys cookies coconut,Snacks


In [84]:
# Compare predicted vs actual category, count how many matched
correct = (bigger_sample_clean["predicted_category"] == bigger_sample_clean["main_category"]).sum()
total = len(bigger_sample_clean)
print(f"{correct} out of {total} correct = {correct/total:.2%}")

15 out of 28 correct = 53.57%


### Result: 53.33% accuracy (16/30 correct) on real OCR text vs. 83% on clean text — a legitimate, explainable finding about real-world OCR noise degrading model performance


# Transformer Model

In [35]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch

# Load the tokenizer — converts text to token IDs DistilBERT understands
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Load the pretrained model — this is the neural network with all its learned weights
model_t = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Set model to evaluation mode — tells it we're using it for inference, not training
model_t.eval()

print("Model and tokenizer loaded successfully!")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 2302.56it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and tokenizer loaded successfully!


In [76]:
# Let's see what the tokenizer does to a sample product name
sample_text = "Organic Dark Chocolate Bar"

# Tokenizer converts the text into token IDs — numbers DistilBERT understands
tokens = tokenizer(sample_text, return_tensors='pt')

print("Token IDs:", tokens['input_ids'])
print("Shape:", tokens['input_ids'].shape)

Token IDs: tensor([[ 101, 7554, 2601, 7967, 3347,  102]])
Shape: torch.Size([1, 6])


In [79]:
import numpy as np

def get_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                          max_length=64, return_tensors='pt')
        with torch.no_grad():
            output = model_t(**encoded)  # using model_t now, not model
        cls_embeddings = output.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_embeddings)
        if (i // batch_size) % 10 == 0:
            print(f"Processed {i}/{len(texts)} products...")
    return np.vstack(all_embeddings)

texts = df['product_name'].fillna('').astype(str).tolist()
print("Generating embeddings... this will take a few minutes")
X_embeddings = get_embeddings(texts)
print(f"Done! Embedding shape: {X_embeddings.shape}")

Generating embeddings... this will take a few minutes
Processed 0/6033 products...
Processed 320/6033 products...
Processed 640/6033 products...
Processed 960/6033 products...
Processed 1280/6033 products...
Processed 1600/6033 products...
Processed 1920/6033 products...
Processed 2240/6033 products...
Processed 2560/6033 products...
Processed 2880/6033 products...
Processed 3200/6033 products...
Processed 3520/6033 products...
Processed 3840/6033 products...
Processed 4160/6033 products...
Processed 4480/6033 products...
Processed 4800/6033 products...
Processed 5120/6033 products...
Processed 5440/6033 products...
Processed 5760/6033 products...
Done! Embedding shape: (6033, 768)


In [85]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split embeddings into 80% train, 20% test
y = df["main_category"]
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42
)

# Train Logistic Regression on the 768-number embeddings
lr_bert = LogisticRegression(max_iter=1000)
lr_bert.fit(X_train_bert, y_train_bert)

# Test and print accuracy
y_pred_bert = lr_bert.predict(X_test_bert)
bert_accuracy = accuracy_score(y_test_bert, y_pred_bert)

print(f"TF-IDF Accuracy:   83%")
print(f"DistilBERT Accuracy: {bert_accuracy:.2%}")
print("\n", classification_report(y_test_bert, y_pred_bert))

TF-IDF Accuracy:   83%
DistilBERT Accuracy: 75.81%

                                  precision    recall  f1-score   support

                      Beverages       0.87      0.80      0.83       142
                     Condiments       0.78      0.83      0.80       108
                        Dairies       0.77      0.74      0.76        82
                       Desserts       0.95      0.82      0.88        68
                   Frozen foods       0.33      0.05      0.08        21
                          Meals       0.60      0.50      0.55        24
       Meats and their products       0.68      0.47      0.56        36
Plant-based foods and beverages       0.70      0.78      0.74       366
                         Snacks       0.75      0.78      0.77       348
                     Sweeteners       1.00      0.58      0.74        12

                       accuracy                           0.76      1207
                      macro avg       0.74      0.64      0.67      1

C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [86]:
print(bigger_sample_clean[["product_name", "main_category", "ocr_text"]].head())
print(f"Rows: {len(bigger_sample_clean)}")


                             product_name                    main_category  \
9436             Pistachios Salt & Pepper  Plant-based foods and beverages   
11787                        Candied yams  Plant-based foods and beverages   
11498           Sharp cheddar cheese ball                          Dairies   
13454           Clabber Girl, Corn Starch  Plant-based foods and beverages   
11036  Crunchy Granola Bars Oats 'n Honey                           Snacks   

                                            ocr_text  
9436                                   Salt & Pepper  
11787  CANDIED YMS CUt SWEET POTATOES SYRUP SIMMERED  
11498                        SPREADABLE C 0 'CHEDDAR  
13454              CLABBER GIRL CORN STARCH THICKENS  
11036                          CRUNCHY Oats Honey 98  
Rows: 28


In [87]:
# Get the OCR texts and real categories
ocr_texts = bigger_sample_clean["ocr_text"].fillna('').astype(str).tolist()
ocr_true_labels = bigger_sample_clean["main_category"].tolist()

# Generate DistilBERT embeddings for the OCR text
print("Generating embeddings for OCR text...")
X_ocr_embeddings = get_embeddings(ocr_texts)

# Use our already trained DistilBERT classifier to predict
ocr_bert_predictions = lr_bert.predict(X_ocr_embeddings)

# Count how many were correct
correct = sum(p == t for p, t in zip(ocr_bert_predictions, ocr_true_labels))
total = len(ocr_true_labels)

print(f"\nResults on OCR text:")
print(f"TF-IDF on OCR text:     53%")
print(f"DistilBERT on OCR text: {correct/total:.2%}")

Generating embeddings for OCR text...
Processed 0/28 products...

Results on OCR text:
TF-IDF on OCR text:     53%
DistilBERT on OCR text: 50.00%


## Stage 4: DistilBERT Transformer Model

### What I did
Instead of counting words like TF-IDF does, I used a pretrained transformer model 
(DistilBERT) to convert each product name into 768 numbers that capture meaning. 
I then trained a Logistic Regression classifier on top of those numbers.

### Results

| Method | Clean Text | OCR Text |
|---|---|---|
| TF-IDF + Logistic Regression | 83% | 53% |
| DistilBERT + Logistic Regression | 75% | 50% |

### What I found
TF-IDF beat DistilBERT on this dataset. Product names are short and keyword-heavy 
— "yogurt", "chocolate", "water" already tell you the category directly. 
DistilBERT's contextual understanding adds no real advantage here.

Neither model handles noisy OCR text well. The bottleneck is input quality, 
not the classifier — clean text gives 83%, garbled OCR text gives 53%.

A fine-tuned DistilBERT trained specifically on food data would likely do better, 
but that needs more compute than this project scope allows.

In [88]:
def predict_with_confidence(text, top_n=3):
    # Convert input text to TF-IDF numbers
    text_tfidf = vectorizer.transform([text])
    
    # Get probability scores for all 10 categories
    probabilities = model.predict_proba(text_tfidf)[0]
    
    # Get category names in the same order as probabilities
    categories = model.classes_
    
    # Pair each category with its probability and sort highest first
    ranked = sorted(zip(categories, probabilities), key=lambda x: x[1], reverse=True)
    
    print(f"Input: {text}")
    print(f"Top {top_n} predictions:")
    for i, (category, prob) in enumerate(ranked[:top_n]):
        print(f"  {i+1}. {category}: {prob:.1%}")

# Test it on a few examples
predict_with_confidence("Organic Dark Chocolate Bar")
print()
predict_with_confidence("Sparkling Mineral Water")
print()
predict_with_confidence("Low Fat Greek Yogurt")

Input: Organic Dark Chocolate Bar
Top 3 predictions:
  1. Snacks: 84.1%
  2. Plant-based foods and beverages: 9.5%
  3. Desserts: 1.3%

Input: Sparkling Mineral Water
Top 3 predictions:
  1. Beverages: 91.7%
  2. Plant-based foods and beverages: 3.1%
  3. Snacks: 1.7%

Input: Low Fat Greek Yogurt
Top 3 predictions:
  1. Dairies: 93.2%
  2. Snacks: 1.6%
  3. Beverages: 1.2%


In [90]:
import os
image_files = os.listdir('openfoodFACT_images')
print(image_files)

['.ipynb_checkpoints', '10339120577.jpg', '104364.jpg', '11110004543.jpg', '11110507198.jpg', '11110738592.jpg', '11110874467.jpg', '11152259901.jpg', '11152431680.jpg', '11210008250.jpg', '11210008328.jpg', '11228000758.jpg', '11300385339.jpg', '11693.jpg', '11826800071.jpg', '120241.jpg', '120609.jpg', '122580.jpg', '12546011662.jpg', '12822009222.jpg', '130530.jpg', '13170792026.jpg', '13562000555.jpg', '13971000214.jpg', '14100095279.jpg', '14113913256.jpg', '14683003708.jpg', '148849.jpg', '15109001223.jpg', '16000159105.jpg', '16000159501.jpg', '16000487598.jpg', '161954.jpg', '16229917647.jpg', '16300165752.jpg', '168526.jpg', '17003135844.jpg', '17077101325.jpg', '17082881656.jpg', '17400118501.jpg', '17600043191.jpg', '178631.jpg', '17869727771.jpg', '19000085047.jpg', '19063005006.jpg', '19458000180.jpg', '19900003851.jpg', '2000000707.jpg', '20000104195.jpg', '20662000200.jpg', '20735092729.jpg', '20735096574.jpg', '21130480814.jpg', '25157.jpg', '76715.jpg', '9300000659.jpg

In [91]:
def predict_from_image(image_path):
    # Step 1 — Run OCR on the image
    result = reader.readtext(image_path)
    
    # Step 2 — Filter by confidence threshold
    filtered_text = [d[1] for d in result if d[2] >= 0.65]
    extracted_text = " ".join(filtered_text)
    
    if not extracted_text.strip():
        print("No text detected in image")
        return
    
    print(f"OCR extracted: {extracted_text}")
    
    # Step 3 — Predict with confidence scores
    predict_with_confidence(extracted_text)

# Test on one of your saved product images
import os

# Only pick files that end with .jpg, skipping folders like .ipynb_checkpoints
image_files = [f for f in os.listdir('openfoodFACT_images') if f.endswith('.jpg')]

test_image = os.path.join('openfoodFACT_images', image_files[0])
print(f"Testing on: {image_files[0]}\n")
predict_from_image(test_image)

Testing on: 10339120577.jpg



C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


OCR extracted: Sorrel Ginger
Input: Sorrel Ginger
Top 3 predictions:
  1. Plant-based foods and beverages: 32.3%
  2. Snacks: 26.9%
  3. Beverages: 15.2%


In [92]:
# Test on first 5 images and see the range of confidence scores
for filename in image_files[:5]:
    image_path = os.path.join('openfoodFACT_images', filename)
    print(f"Image: {filename}")
    predict_from_image(image_path)
    print()

Image: 10339120577.jpg


C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


OCR extracted: Sorrel Ginger
Input: Sorrel Ginger
Top 3 predictions:
  1. Plant-based foods and beverages: 32.3%
  2. Snacks: 26.9%
  3. Beverages: 15.2%

Image: 104364.jpg


C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


OCR extracted: M&s BBQ BEEF JERKY Hich NICE To Meatvou
Input: M&s BBQ BEEF JERKY Hich NICE To Meatvou
Top 3 predictions:
  1. Snacks: 65.5%
  2. Meats and their products: 11.9%
  3. Condiments: 7.6%

Image: 11110004543.jpg


...

OCR extracted: VANILLA BEAN VANILLA BEAN
Input: VANILLA BEAN VANILLA BEAN
Top 3 predictions:
  1. Snacks: 43.1%
  2. Plant-based foods and beverages: 16.7%
  3. Desserts: 12.0%

Image: 11110507198.jpg


C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


OCR extracted: (DELUXE ICE CREAM COOKIES N' CREAM
Input: (DELUXE ICE CREAM COOKIES N' CREAM
Top 3 predictions:
  1. Desserts: 97.3%
  2. Snacks: 1.8%
  3. Dairies: 0.3%

Image: 11110738592.jpg


C:\Users\jayak\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


OCR extracted: milk dutch hot cocoa chocolate
Input: milk dutch hot cocoa chocolate
Top 3 predictions:
  1. Beverages: 69.9%
  2. Snacks: 16.5%
  3. Plant-based foods and beverages: 3.5%



## Stage 5: Confidence-Ranked Predictions

### What I added
Instead of a single flat prediction, the model now outputs the top 3 category 
guesses ranked by confidence score. This makes the tool more honest — 
a 97% confidence score means trust the result, a 32% score means 
the OCR didn't extract enough text to be sure.

### Examples
- "DELUXE ICE CREAM COOKIES N' CREAM" → Desserts 97.3% ✓
- "milk dutch hot cocoa chocolate" → Beverages 69.9% ✓  
- "Sorrel Ginger" → Plant-based foods 32.3% (low confidence, little text extracted)

### Key insight
Confidence scores expose the real bottleneck — it's not the classifier 
that fails, it's poor OCR extraction on unclear images.

In [93]:
import os
print(os.getcwd())

C:\Users\jayak\MY PYTHONNNNNNNN
